<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Run_Theory_ve_Drought_Characteristics_Box_Plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RUN THEORY GRAFİKLERİ

In [ ]:
pip install pandas matplotlib numpy seaborn

In [ ]:
"""
Figure 3 — Drought Characteristics Box Plots
6 panels: Duration and Severity for SPI, SDI, RDI
By station, colored by index, Scale-6 primary + inset Scale-3/12

Requirements:
    pip install pandas matplotlib numpy seaborn
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_DIR  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run Theory_Makale/Figures"
THRESHOLD   = -0.5

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# DATA
# ============================================================
df = pd.read_csv(EVENTS_FILE)
df["Peak"] = df["Peak"].abs()
df05 = df[df["Threshold"] == THRESHOLD].copy()

STATIONS = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]
INDICES  = ["SPI", "SDI", "RDI"]
SCALES   = [3, 6, 12]

# Color palette — consistent with other figures
IDX_COLORS = {
    "SPI": {"main": "#1f6f2e", "light": "#74c476", "dark": "#00441b"},
    "SDI": {"main": "#08519c", "light": "#6baed6", "dark": "#08306b"},
    "RDI": {"main": "#8c2d04", "light": "#fc8d59", "dark": "#67000d"},
}

STATION_SHORT = {
    "Kastamonu": "KST", "Sivas": "SVS",
    "Kayseri":   "KSR", "Yozgat": "YZG",
    "Kirikkale": "KRK",
}

# ============================================================
# FIGURE 3a — Scale-6, Duration & Severity by Station
# Main publication figure
# ============================================================
def make_figure3a():
    scale = 6
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor("white")

    variables = ["Duration", "Severity"]
    ylabels   = ["Duration (months)", "Severity (cumulative deficit)"]
    ytitles   = [
        "(a) Drought Duration — Scale-6",
        "(b) Drought Severity — Scale-6",
    ]

    for ax, var, ylab, ytitle in zip(axes, variables, ylabels, ytitles):
        x_positions = []
        box_data    = []
        box_colors  = []
        x_labels    = []

        x = 0
        for si, station in enumerate(STATIONS):
            for idx in INDICES:
                sub = df05[
                    (df05["Station"]==station) &
                    (df05["Index"]==idx) &
                    (df05["Scale"]==scale)
                ][var].dropna().values

                if len(sub) < 3:
                    x += 1
                    continue

                x_positions.append(x)
                box_data.append(sub)
                box_colors.append(IDX_COLORS[idx]["main"])
                x_labels.append(f"{STATION_SHORT[station]}\n{idx}")
                x += 1
            x += 0.8   # gap between stations

        # Draw boxes
        bp = ax.boxplot(
            box_data,
            positions=x_positions,
            widths=0.6,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker="o", markersize=3,
                            markerfacecolor="gray", alpha=0.5,
                            linestyle="none"),
            medianprops=dict(color="white", linewidth=2),
            whiskerprops=dict(linewidth=1.2, color="black"),
            capprops=dict(linewidth=1.5, color="black"),
            boxprops=dict(linewidth=1.2),
        )

        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.82)

        # Station separator lines
        x_sep = 0
        for si in range(len(STATIONS)-1):
            x_sep += 3 + 0.8
            ax.axvline(x_sep - 0.4, color="gray",
                       linewidth=0.8, linestyle="--", alpha=0.5)

        # Station labels at top
        x_st = 0
        for si, station in enumerate(STATIONS):
            n_idx = sum(1 for idx in INDICES
                       if len(df05[(df05["Station"]==station) &
                                   (df05["Index"]==idx) &
                                   (df05["Scale"]==scale)]) >= 3)
            mid = x_st + (n_idx - 1) / 2
            ax.text(mid, ax.get_ylim()[1] if ax.get_ylim()[1] > 1 else 1,
                    station, ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color="black")
            x_st += n_idx + 0.8

        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=7.5, rotation=0)
        ax.set_ylabel(ylab, fontsize=10)
        ax.set_title(ytitle, fontsize=11, fontweight="bold", pad=8)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_facecolor("#fafafa")

        # Re-draw station labels now that y limits are set
        ymax = ax.get_ylim()[1]
        x_st = 0
        for si, station in enumerate(STATIONS):
            idx_available = [idx for idx in INDICES
                            if len(df05[(df05["Station"]==station) &
                                        (df05["Index"]==idx) &
                                        (df05["Scale"]==scale)]) >= 3]
            if not idx_available:
                continue
            n_idx = len(idx_available)
            mid = x_st + (n_idx - 1) / 2
            ax.text(mid, ymax * 0.97, station,
                    ha="center", va="top",
                    fontsize=9, fontweight="bold",
                    color="#333333",
                    bbox=dict(boxstyle="round,pad=0.2",
                              facecolor="white", alpha=0.7,
                              edgecolor="lightgray"))
            x_st += n_idx + 0.8

    # Legend
    legend_patches = [
        mpatches.Patch(color=IDX_COLORS[idx]["main"],
                       alpha=0.82, label=idx)
        for idx in INDICES
    ]
    axes[0].legend(handles=legend_patches, loc="upper right",
                   fontsize=9, title="Index", framealpha=0.9)

    fig.suptitle(
        "Figure 3. Drought Characteristics at Scale-6 across Kızılırmak Basin Stations\n"
        "Threshold = −0.5  |  Box: IQR  |  Line: median  |  Whiskers: 1.5×IQR",
        fontsize=11, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "Figure3a_DroughtCharacteristics_Scale6.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ============================================================
# FIGURE 3b — All scales, Duration & Severity (6 panels)
# Full version matching reference paper style
# ============================================================
def make_figure3b():
    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor("white")
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.38, wspace=0.28)

    panels = [
        ("Duration", 3,  "SPI", "(a) SPI Duration"),
        ("Duration", 6,  "SPI", "(b) SPI Duration"),
        ("Duration", 12, "SPI", "(c) SPI Duration"),
        ("Severity", 3,  "SPI", "(d) SPI Severity"),
        ("Severity", 6,  "SPI", "(e) SPI Severity"),
        ("Severity", 12, "SPI", "(f) SPI Severity"),
    ]

    # Actually: 6 panels = 2 rows (Duration, Severity) x 3 cols (SPI,SDI,RDI)
    # for Scale-6 only, comparing all 3 indices across stations
    panel_defs = [
        # (row, col, variable, index, title, scale)
        (0, 0, "Duration", "SPI", "(a) Meteorological Drought\nDuration (SPI-6)", 6),
        (0, 1, "Duration", "SDI", "(b) Hydrological Drought\nDuration (SDI-6)",  6),
        (0, 2, "Duration", "RDI", "(c) Reconnaissance Drought\nDuration (RDI-6)", 6),
        (1, 0, "Severity", "SPI", "(d) Meteorological Drought\nSeverity (SPI-6)", 6),
        (1, 1, "Severity", "SDI", "(e) Hydrological Drought\nSeverity (SDI-6)",  6),
        (1, 2, "Severity", "RDI", "(f) Reconnaissance Drought\nSeverity (RDI-6)", 6),
    ]

    for row, col, var, idx, title, scale in panel_defs:
        ax = fig.add_subplot(gs[row, col])

        data_by_station = []
        labels = []
        n_list = []

        for station in STATIONS:
            sub = df05[
                (df05["Station"]==station) &
                (df05["Index"]==idx) &
                (df05["Scale"]==scale)
            ][var].dropna().values
            data_by_station.append(sub if len(sub) >= 3 else np.array([np.nan]))
            labels.append(STATION_SHORT[station])
            n_list.append(len(sub) if len(sub) >= 3 else 0)

        positions = list(range(1, len(STATIONS)+1))
        color_main  = IDX_COLORS[idx]["main"]
        color_light = IDX_COLORS[idx]["light"]

        bp = ax.boxplot(
            data_by_station,
            positions=positions,
            widths=0.55,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker="o", markersize=3.5,
                            markerfacecolor=color_light,
                            markeredgecolor=color_main,
                            alpha=0.6, linestyle="none"),
            medianprops=dict(color="white", linewidth=2.5),
            whiskerprops=dict(linewidth=1.3, color="#333333"),
            capprops=dict(linewidth=1.8, color="#333333"),
            boxprops=dict(linewidth=1.3),
            meanprops=dict(marker="D", markersize=5,
                           markerfacecolor="white",
                           markeredgecolor="black"),
            showmeans=True,
        )

        for patch in bp["boxes"]:
            patch.set_facecolor(color_main)
            patch.set_alpha(0.80)

        # n labels below each box
        for xi, (pos, n) in enumerate(zip(positions, n_list)):
            ax.text(pos, ax.get_ylim()[0],
                    f"n={n}", ha="center", va="top",
                    fontsize=7, color="#555555")

        ax.set_xticks(positions)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_title(title, fontsize=10, fontweight="bold", pad=6)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_facecolor("#fafafa")

        if col == 0:
            ylab = "Duration (months)" if var=="Duration" else "Severity"
            ax.set_ylabel(ylab, fontsize=9)

        # n labels (reposition after ylim is set)
        ymin = ax.get_ylim()[0]
        for pos, n in zip(positions, n_list):
            ax.text(pos, ymin * 1.02 if ymin > 0 else ymin - ymin*0.05,
                    f"n={n}", ha="center", va="top",
                    fontsize=7, color="#666666")

    fig.suptitle(
        "Figure 2. Drought Duration and Severity by Station — Scale-6, Threshold = −0.5\n"
        "(a–c) Duration; (d–f) Severity for SPI, SDI, and RDI\n"
        "Box: IQR; line: median; diamond: mean; whiskers: 1.5×IQR; dots: outliers",
        fontsize=11, fontweight="bold", y=1.01
    )

    out = os.path.join(OUTPUT_DIR, "Figure3b_DroughtCharacteristics_6panel.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ============================================================
# FIGURE 3c — All 3 scales comparison (SPI only, then SDI, RDI)
# 3x2 grid: rows=Scale(3,6,12), cols=Duration,Severity
# ============================================================
def make_figure3c():
    for idx in INDICES:
        fig, axes = plt.subplots(3, 2, figsize=(14, 14))
        fig.patch.set_facecolor("white")

        color_main  = IDX_COLORS[idx]["main"]
        color_light = IDX_COLORS[idx]["light"]

        for ri, scale in enumerate(SCALES):
            for ci, var in enumerate(["Duration","Severity"]):
                ax = axes[ri, ci]

                data = []
                labels = []
                n_list = []
                for station in STATIONS:
                    sub = df05[
                        (df05["Station"]==station) &
                        (df05["Index"]==idx) &
                        (df05["Scale"]==scale)
                    ][var].dropna().values
                    data.append(sub if len(sub)>=3 else np.array([np.nan]))
                    labels.append(STATION_SHORT[station])
                    n_list.append(len(sub) if len(sub)>=3 else 0)

                bp = ax.boxplot(
                    data,
                    patch_artist=True,
                    showfliers=True,
                    flierprops=dict(marker="o", markersize=3,
                                    markerfacecolor=color_light,
                                    alpha=0.5, linestyle="none"),
                    medianprops=dict(color="white", linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.5),
                    meanprops=dict(marker="D", markersize=5,
                                   markerfacecolor="white",
                                   markeredgecolor="black"),
                    showmeans=True,
                )
                for patch in bp["boxes"]:
                    patch.set_facecolor(color_main)
                    patch.set_alpha(0.80)

                ax.set_xticklabels(labels, fontsize=9)
                ax.set_title(
                    f"{idx}-{scale}  {var}",
                    fontsize=10, fontweight="bold"
                )
                ax.grid(axis="y", linestyle="--", alpha=0.4)
                ax.set_facecolor("#fafafa")

                if ci == 0:
                    ax.set_ylabel("Duration (months)", fontsize=9)
                else:
                    ax.set_ylabel("Severity", fontsize=9)

                # n labels
                for xi, n in enumerate(n_list, start=1):
                    ymin = ax.get_ylim()[0]
                    ax.text(xi, ymin, f"n={n}",
                            ha="center", va="top",
                            fontsize=7, color="#666666")

        fig.suptitle(
            f"Figure 3{idx}. {idx} Drought Duration and Severity — "
            f"All Scales (3, 6, 12 months)\n"
            f"Threshold = −0.5  |  Diamond: mean  |  Line: median",
            fontsize=11, fontweight="bold"
        )
        plt.tight_layout()
        out = os.path.join(OUTPUT_DIR, f"Figure3c_{idx}_AllScales.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  Saved -> {out}")


# ============================================================
# MAIN
# ============================================================
print("="*55)
print("  Figure 3 — Drought Characteristics")
print("="*55 + "\n")

print("[Fig 3a] Scale-6 combined (SPI+SDI+RDI by station)...")
make_figure3a()

print("[Fig 3b] 6-panel (Duration+Severity x SPI+SDI+RDI)...")
make_figure3b()

print("[Fig 3c] All scales per index...")
make_figure3c()

print("\nAll Figure 3 variants completed!")
print(f"Output -> {OUTPUT_DIR}")

In [ ]:
"""
Figure 3 — Drought Characteristics Box Plots
6 panels: Duration and Severity for SPI, SDI, RDI
By station, colored by index, Scale-6 primary + inset Scale-3/12

Requirements:
    pip install pandas matplotlib numpy seaborn
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_DIR  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/Figures2"
THRESHOLD   = -0.5

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# DATA
# ============================================================
df = pd.read_csv(EVENTS_FILE)
df["Peak"] = df["Peak"].abs()
df05 = df[df["Threshold"] == THRESHOLD].copy()

STATIONS = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]
INDICES  = ["SPI", "SDI", "RDI"]
SCALES   = [3, 6, 12]

# Color palette — consistent with other figures
IDX_COLORS = {
    "SPI": {"main": "#1f6f2e", "light": "#74c476", "dark": "#00441b"},
    "SDI": {"main": "#08519c", "light": "#6baed6", "dark": "#08306b"},
    "RDI": {"main": "#8c2d04", "light": "#fc8d59", "dark": "#67000d"},
}

STATION_SHORT = {
    "Kastamonu": "KST", "Sivas": "SVS",
    "Kayseri":   "KSR", "Yozgat": "YZG",
    "Kirikkale": "KRK",
}

# ============================================================
# FIGURE 3a — Scale-6, Duration & Severity by Station
# Main publication figure
# ============================================================
def make_figure3a():
    scale = 6
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor("white")

    variables = ["Duration", "Severity"]
    ylabels   = ["Duration (months)", "Severity (cumulative deficit)"]
    ytitles   = [
        "(a) Drought Duration — Scale-6",
        "(b) Drought Severity — Scale-6",
    ]

    for ax, var, ylab, ytitle in zip(axes, variables, ylabels, ytitles):
        x_positions = []
        box_data    = []
        box_colors  = []
        x_labels    = []

        x = 0
        for si, station in enumerate(STATIONS):
            for idx in INDICES:
                sub = df05[
                    (df05["Station"]==station) &
                    (df05["Index"]==idx) &
                    (df05["Scale"]==scale)
                ][var].dropna().values

                if len(sub) < 3:
                    x += 1
                    continue

                x_positions.append(x)
                box_data.append(sub)
                box_colors.append(IDX_COLORS[idx]["main"])
                x_labels.append(f"{STATION_SHORT[station]}\n{idx}")
                x += 1
            x += 0.8   # gap between stations

        # Draw boxes
        bp = ax.boxplot(
            box_data,
            positions=x_positions,
            widths=0.6,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker="o", markersize=3,
                            markerfacecolor="gray", alpha=0.5,
                            linestyle="none"),
            medianprops=dict(color="white", linewidth=2),
            whiskerprops=dict(linewidth=1.2, color="black"),
            capprops=dict(linewidth=1.5, color="black"),
            boxprops=dict(linewidth=1.2),
        )

        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.82)

        # Station separator lines
        x_sep = 0
        for si in range(len(STATIONS)-1):
            x_sep += 3 + 0.8
            ax.axvline(x_sep - 0.4, color="gray",
                       linewidth=0.8, linestyle="--", alpha=0.5)

        # Station labels at top
        x_st = 0
        for si, station in enumerate(STATIONS):
            n_idx = sum(1 for idx in INDICES
                       if len(df05[(df05["Station"]==station) &
                                   (df05["Index"]==idx) &
                                   (df05["Scale"]==scale)]) >= 3)
            mid = x_st + (n_idx - 1) / 2
            ax.text(mid, ax.get_ylim()[1] if ax.get_ylim()[1] > 1 else 1,
                    station, ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color="black")
            x_st += n_idx + 0.8

        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=7.5, rotation=0)
        ax.set_ylabel(ylab, fontsize=10)
        ax.set_title(ytitle, fontsize=11, fontweight="bold", pad=8)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_facecolor("#fafafa")

        # Re-draw station labels now that y limits are set
        ymax = ax.get_ylim()[1]
        x_st = 0
        for si, station in enumerate(STATIONS):
            idx_available = [idx for idx in INDICES
                            if len(df05[(df05["Station"]==station) &
                                        (df05["Index"]==idx) &
                                        (df05["Scale"]==scale)]) >= 3]
            if not idx_available:
                continue
            n_idx = len(idx_available)
            mid = x_st + (n_idx - 1) / 2
            ax.text(mid, ymax * 0.97, station,
                    ha="center", va="top",
                    fontsize=9, fontweight="bold",
                    color="#333333",
                    bbox=dict(boxstyle="round,pad=0.2",
                              facecolor="white", alpha=0.7,
                              edgecolor="lightgray"))
            x_st += n_idx + 0.8

    # Legend
    legend_patches = [
        mpatches.Patch(color=IDX_COLORS[idx]["main"],
                       alpha=0.82, label=idx)
        for idx in INDICES
    ]
    axes[0].legend(handles=legend_patches, loc="upper right",
                   fontsize=9, title="Index", framealpha=0.9)

    fig.suptitle(
        "Figure 3. Drought Characteristics at Scale-6 across Kızılırmak Basin Stations\n"
        "Threshold = −0.5  |  Box: IQR  |  Line: median  |  Whiskers: 1.5×IQR",
        fontsize=11, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "Figure3a_DroughtCharacteristics_Scale6.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ============================================================
# FIGURE 3b — All scales, Duration & Severity (6 panels)
# Full version matching reference paper style
# ============================================================
def make_figure3b():
    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor("white")
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.38, wspace=0.28)

    panels = [
        ("Duration", 3,  "SPI", "(a) SPI Duration"),
        ("Duration", 6,  "SPI", "(b) SPI Duration"),
        ("Duration", 12, "SPI", "(c) SPI Duration"),
        ("Severity", 3,  "SPI", "(d) SPI Severity"),
        ("Severity", 6,  "SPI", "(e) SPI Severity"),
        ("Severity", 12, "SPI", "(f) SPI Severity"),
    ]

    # Actually: 6 panels = 2 rows (Duration, Severity) x 3 cols (SPI,SDI,RDI)
    # for Scale-6 only, comparing all 3 indices across stations
    panel_defs = [
        # (row, col, variable, index, title, scale)
        (0, 0, "Duration", "SPI", "(a) Meteorological Drought\nDuration (SPI-6)", 6),
        (0, 1, "Duration", "SDI", "(b) Hydrological Drought\nDuration (SDI-6)",  6),
        (0, 2, "Duration", "RDI", "(c) Reconnaissance Drought\nDuration (RDI-6)", 6),
        (1, 0, "Severity", "SPI", "(d) Meteorological Drought\nSeverity (SPI-6)", 6),
        (1, 1, "Severity", "SDI", "(e) Hydrological Drought\nSeverity (SDI-6)",  6),
        (1, 2, "Severity", "RDI", "(f) Reconnaissance Drought\nSeverity (RDI-6)", 6),
    ]

    for row, col, var, idx, title, scale in panel_defs:
        ax = fig.add_subplot(gs[row, col])

        data_by_station = []
        labels = []
        n_list = []

        for station in STATIONS:
            sub = df05[
                (df05["Station"]==station) &
                (df05["Index"]==idx) &
                (df05["Scale"]==scale)
            ][var].dropna().values
            data_by_station.append(sub if len(sub) >= 3 else np.array([np.nan]))
            labels.append(STATION_SHORT[station])
            n_list.append(len(sub) if len(sub) >= 3 else 0)

        positions = list(range(1, len(STATIONS)+1))
        color_main  = IDX_COLORS[idx]["main"]
        color_light = IDX_COLORS[idx]["light"]

        bp = ax.boxplot(
            data_by_station,
            positions=positions,
            widths=0.55,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker="o", markersize=3.5,
                            markerfacecolor=color_light,
                            markeredgecolor=color_main,
                            alpha=0.6, linestyle="none"),
            medianprops=dict(color="white", linewidth=2.5),
            whiskerprops=dict(linewidth=1.3, color="#333333"),
            capprops=dict(linewidth=1.8, color="#333333"),
            boxprops=dict(linewidth=1.3),
            meanprops=dict(marker="D", markersize=5,
                           markerfacecolor="white",
                           markeredgecolor="black"),
            showmeans=True,
        )

        for patch in bp["boxes"]:
            patch.set_facecolor(color_main)
            patch.set_alpha(0.80)

        # n labels below each box
        for xi, (pos, n) in enumerate(zip(positions, n_list)):
            ax.text(pos, ax.get_ylim()[0],
                    f"n={n}", ha="center", va="top",
                    fontsize=7, color="#555555")

        ax.set_xticks(positions)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_title(title, fontsize=10, fontweight="bold", pad=6)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_facecolor("#fafafa")

        if col == 0:
            ylab = "Duration (months)" if var=="Duration" else "Severity"
            ax.set_ylabel(ylab, fontsize=9)

        # n labels (reposition after ylim is set)
        ymin = ax.get_ylim()[0]
        for pos, n in zip(positions, n_list):
            ax.text(pos, ymin * 1.02 if ymin > 0 else ymin - ymin*0.05,
                    f"n={n}", ha="center", va="top",
                    fontsize=7, color="#666666")

    fig.suptitle(
        "Figure 3. Drought Duration and Severity by Station — Scale-6, Threshold = −0.5\n"
        "(a–c) Duration; (d–f) Severity for SPI, SDI, and RDI\n"
        "Box: IQR; line: median; diamond: mean; whiskers: 1.5×IQR; dots: outliers",
        fontsize=11, fontweight="bold", y=1.01
    )

    out = os.path.join(OUTPUT_DIR, "Figure3b_DroughtCharacteristics_6panel.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ============================================================
# FIGURE 3c — All 3 scales comparison (SPI only, then SDI, RDI)
# 3x2 grid: rows=Scale(3,6,12), cols=Duration,Severity
# ============================================================
def make_figure3c():
    for idx in INDICES:
        fig, axes = plt.subplots(3, 2, figsize=(14, 14))
        fig.patch.set_facecolor("white")

        color_main  = IDX_COLORS[idx]["main"]
        color_light = IDX_COLORS[idx]["light"]

        for ri, scale in enumerate(SCALES):
            for ci, var in enumerate(["Duration","Severity"]):
                ax = axes[ri, ci]

                data = []
                labels = []
                n_list = []
                for station in STATIONS:
                    sub = df05[
                        (df05["Station"]==station) &
                        (df05["Index"]==idx) &
                        (df05["Scale"]==scale)
                    ][var].dropna().values
                    data.append(sub if len(sub)>=3 else np.array([np.nan]))
                    labels.append(STATION_SHORT[station])
                    n_list.append(len(sub) if len(sub)>=3 else 0)

                bp = ax.boxplot(
                    data,
                    patch_artist=True,
                    showfliers=True,
                    flierprops=dict(marker="o", markersize=3,
                                    markerfacecolor=color_light,
                                    alpha=0.5, linestyle="none"),
                    medianprops=dict(color="white", linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.5),
                    meanprops=dict(marker="D", markersize=5,
                                   markerfacecolor="white",
                                   markeredgecolor="black"),
                    showmeans=True,
                )
                for patch in bp["boxes"]:
                    patch.set_facecolor(color_main)
                    patch.set_alpha(0.80)

                ax.set_xticklabels(labels, fontsize=9)
                ax.set_title(
                    f"{idx}-{scale}  {var}",
                    fontsize=10, fontweight="bold"
                )
                ax.grid(axis="y", linestyle="--", alpha=0.4)
                ax.set_facecolor("#fafafa")

                if ci == 0:
                    ax.set_ylabel("Duration (months)", fontsize=9)
                else:
                    ax.set_ylabel("Severity", fontsize=9)

                # n labels
                for xi, n in enumerate(n_list, start=1):
                    ymin = ax.get_ylim()[0]
                    ax.text(xi, ymin, f"n={n}",
                            ha="center", va="top",
                            fontsize=7, color="#666666")

        fig.suptitle(
            f"Figure 3{idx}. {idx} Drought Duration and Severity — "
            f"All Scales (3, 6, 12 months)\n"
            f"Threshold = −0.5  |  Diamond: mean  |  Line: median",
            fontsize=11, fontweight="bold"
        )
        plt.tight_layout()
        out = os.path.join(OUTPUT_DIR, f"Figure3c_{idx}_AllScales.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  Saved -> {out}")


# ============================================================
# MAIN
# ============================================================
print("="*55)
print("  Figure 3 — Drought Characteristics")
print("="*55 + "\n")

print("[Fig 3a] Scale-6 combined (SPI+SDI+RDI by station)...")
make_figure3a()

print("[Fig 3b] 6-panel (Duration+Severity x SPI+SDI+RDI)...")
make_figure3b()

print("[Fig 3c] All scales per index...")
make_figure3c()

print("\nAll Figure 3 variants completed!")
print(f"Output -> {OUTPUT_DIR}")
"""
Figure 3b — Düzeltilmiş versiyon
Bu kodu Colab'da yeni bir hücreye yapıştırın ve çalıştırın.
Üstteki koddan sonra çalıştırılmalı (df05, STATIONS vb. tanımlı olmalı)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Ayarları buradan güncelleyin ──────────────────────────
OUTPUT_DIR  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IDX_COLORS = {
    "SPI": {"main": "#1f6f2e", "light": "#74c476"},
    "SDI": {"main": "#08519c", "light": "#6baed6"},
    "RDI": {"main": "#8c2d04", "light": "#fc8d59"},
}
STATION_SHORT = {
    "Kastamonu": "KST", "Sivas": "SVS",
    "Kayseri":   "KSR", "Yozgat": "YZG",
    "Kirikkale": "KRK",
}
# ─────────────────────────────────────────────────────────

panel_defs = [
    (0, 0, "Duration", "SPI", "(a) Meteorological Drought\nDuration (SPI-6)",  6),
    (0, 1, "Duration", "SDI", "(b) Hydrological Drought\nDuration (SDI-6)",    6),
    (0, 2, "Duration", "RDI", "(c) Reconnaissance Drought\nDuration (RDI-6)", 6),
    (1, 0, "Severity", "SPI", "(d) Meteorological Drought\nSeverity (SPI-6)",  6),
    (1, 1, "Severity", "SDI", "(e) Hydrological Drought\nSeverity (SDI-6)",    6),
    (1, 2, "Severity", "RDI", "(f) Reconnaissance Drought\nSeverity (RDI-6)", 6),
]

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor("white")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.32)

for row, col, var, idx, title, scale in panel_defs:
    ax = fig.add_subplot(gs[row, col])

    data_by_station = []
    labels  = []
    n_list  = []

    for station in STATIONS:
        sub = df05[
            (df05["Station"] == station) &
            (df05["Index"]   == idx) &
            (df05["Scale"]   == scale)
        ][var].dropna().values

        data_by_station.append(sub if len(sub) >= 3 else np.array([np.nan]))
        labels.append(STATION_SHORT[station])
        n_list.append(len(sub) if len(sub) >= 3 else 0)

    positions   = list(range(1, len(STATIONS) + 1))
    color_main  = IDX_COLORS[idx]["main"]
    color_light = IDX_COLORS[idx]["light"]

    # SDI Severity → log scale
    use_log = (idx == "SDI" and var == "Severity")

    bp = ax.boxplot(
        data_by_station,
        positions   = positions,
        widths      = 0.55,
        patch_artist= True,
        showfliers  = True,
        flierprops  = dict(marker="o", markersize=3.5,
                           markerfacecolor=color_light,
                           markeredgecolor=color_main,
                           alpha=0.6, linestyle="none"),
        medianprops = dict(color="white",   linewidth=2.5),
        whiskerprops= dict(color="#333333", linewidth=1.3),
        capprops    = dict(color="#333333", linewidth=1.8),
        boxprops    = dict(linewidth=1.3),
        meanprops   = dict(marker="D", markersize=5,
                           markerfacecolor="white",
                           markeredgecolor="black"),
        showmeans   = True,
    )

    for patch in bp["boxes"]:
        patch.set_facecolor(color_main)
        patch.set_alpha(0.82)

    if use_log:
        ax.set_yscale("log")

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=9.5)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.set_facecolor("#fafafa")
    ax.spines[["top","right"]].set_visible(False)

    # Y ekseni etiketi
    if col == 0:
        ylab = "Duration (months)" if var == "Duration" else "Severity"
        ax.set_ylabel(ylab, fontsize=9.5)
    if use_log:
        ax.set_ylabel("Severity (log scale)", fontsize=9.5)

    # n= etiketleri — x ekseninin hemen altına, sabit konumda
    for pos, n in zip(positions, n_list):
        ax.annotate(
            f"n={n}",
            xy=(pos, 0), xycoords=("data", "axes fraction"),
            xytext=(0, -22), textcoords="offset points",
            ha="center", va="top",
            fontsize=7.5, color="#555555",
        )

fig.suptitle(
    "Figure 3. Drought Duration and Severity — Scale-6  |  Threshold = −0.5\n"
    "Box: IQR  |  Line: median  |  Diamond: mean  |  Whiskers: 1.5×IQR",
    fontsize=11, fontweight="bold", y=1.01
)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "Figure3b_fixed.png")
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Kaydedildi → {out}")